In [2]:
# pip install -r requirements.txt

In [3]:
import os
# import pandas as pd
import numpy as np
import json
import sys
import time
from datetime import datetime
import spacy
from sentence_transformers import SentenceTransformer
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from chromadb import Client
from chromadb.config import Settings


/Users/cyrilbenedictlugod/Documents/Misc/random_code_stuff/local-rag/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
np.__version__

'1.26.4'

In [5]:
PERSISTENT_DIRECTORY = './chroma_db'
COLLECTION_NAME = 'local_collection'

# Initialize the console
console = Console()

# Set-up Chroma DB
settings = Settings(persist_directory=PERSISTENT_DIRECTORY, is_persistent=True)

# Initialize the Chroma DB client
client = Client(settings)

# Get or create the index/collection
collection = client.get_or_create_collection(COLLECTION_NAME) # creates at first time/ gets if already exists

### new functions

In [6]:
# Load model



In [ ]:
def load_tracker():
    
    tracker_json_path = 'processed_files.json'

    if os.path.exists(tracker_json_path):
        with open(tracker_json_path, 'r') as f:
            return json.load(f)
    
    else:
        return []

In [8]:
load_tracker()

{}

In [9]:
spacy.load('en_core_web_sm')
import en_core_web_sm
nlp = en_core_web_sm.load()

def chunk_text(text: str, chunk_size=30, overlap=5) -> list:
    
    doc = nlp(text)
    chunks = []
    start = 0
    while start < len(doc):
        end = min(start + chunk_size, len(doc))
        chunks.append(doc[start:end].text)
        start += chunk_size - overlap
    return chunks

In [10]:
model = SentenceTransformer('all-MiniLM-L12-v2')

def get_embedding(chunk):

    embedding = model.encode(chunk).tolist()
    return embedding

In [21]:
def add_vector(chunk_embed, chunk_metadata, chunk, chunk_id):

    collection.add(
        embeddings=[chunk_embed],
        metadatas=[chunk_metadata],
        documents=[chunk],
        ids=[chunk_id],
        )

In [49]:
def delete_vector(file_name, doc_):

    collection.delete(
        where={
            "$and": [
                {"file_name": file_name},  
                {"timestamp": doc_["timestamp"]}
            ]
        }
        )

In [51]:
# test_json = [{'file_name': 'test_doc1.txt',
#              'timestamp': 1740758400.0,
#              'ids': [1, 2, 3, 4, 5]
#              },
#             {'file_name': 'test_doc2.txt',
#             'timestamp': 1740931200.0,
#             'ids': [1, 2, 3]
#             }]

In [52]:
# with open('processed_files.json', 'w') as f:
#     json.dump(test_json, f)

In [53]:
# with open('test.json', 'r') as f:
#     test_json_lst = json.load(f)

In [55]:
def update_files():
        
    tracker_json = load_tracker()
    print(tracker_json)

    with open('processed_files.json', 'w') as f:
        json.dump(tracker_json, f)

    doc_folder = 'docs'

    if os.path.exists(doc_folder):

        # has_update = False

        for file in os.listdir(doc_folder): # iterate through files in folder
            
            file_name = os.path.basename(file)
            file_path = os.path.join(doc_folder, file)

            file_timestamp = os.path.getmtime(file_path)



            with open(file_path, 'r') as f:
                file_content = f.read()
                file_content_chunks = chunk_text(file_content)

            
            id_list = []
            for id_, chunk in enumerate(file_content_chunks):
                id_list.append(id_)
                chunk_embed = get_embedding(chunk)
                # print(chunk_embed)

                chunk_id = f'{file_name}_{id_}'

                chunk_metadata = {'file_name': file_name,
                                  'chunk_number': id_,
                                  'timestamp': file_timestamp,
                                  'preview': chunk[:30]
                                  }
                
                add_vector(chunk_embed, chunk_metadata, chunk, chunk_id)
                console.print(f"[green]Added {chunk_id} - {file_timestamp}[/green]")

            for doc_ in tracker_json:
                if (doc_['file_name'] == file_name) and (doc_['timestamp'] < file_timestamp):
                    doc_['timestamp'] = file_timestamp
                    doc_['ids'] = id_list
                    delete_vector(file_name, doc_)
                    console.print(f"[red]Deleted {file_name} - {doc_['timestamp']}[/red]")
                    # has_update = True

            
            doc_json = {'file_name': file_name,
                        'timestamp': file_timestamp,
                        'ids': id_list
                        }
            
            tracker_json.append(doc_json)

            print(tracker_json)
            

            # else:
                # tracker_json = {}
                # tracker_json['file_name'] = file_name
                # tracker_json['timestamp'] = os.path.getmtime(file)

                # tracker_json['ids'] = # insert ids from embed_chunks here

    # if has_update:
    with open('processed_files.json', 'w') as f:
        json.dump(tracker_json, f)



update_files()

Add of existing embedding ID: doc5.txt_0
Insert of existing embedding ID: doc5.txt_0


[{'file_name': 'doc5.txt', 'timestamp': 1743090118.542274, 'ids': [0, 1, 2]}, {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]}, {'file_name': 'doc1.txt', 'timestamp': 1742099250.607932, 'ids': [0, 1, 2, 3, 4]}, {'file_name': 'doc3.txt', 'timestamp': 1743090121.1379087, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc2.txt', 'timestamp': 1742099152.529095, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc5.txt', 'timestamp': 1743090118.542274, 'ids': [0, 1, 2]}, {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]}, {'file_name': 'doc1.txt', 'timestamp': 1742099250.607932, 'ids': [0, 1, 2, 3, 4]}, {'file_name': 'doc3.txt', 'timestamp': 1743090121.1379087, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc2.txt', 'timestamp': 1742099152.529095, 'ids': [0, 1, 2, 3]}]


Added doc5.txt_0 - 1743090118.542274

Add of existing embedding ID: doc5.txt_1
Insert of existing embedding ID: doc5.txt_1


Added doc5.txt_1 - 1743090118.542274

...

Add of existing embedding ID: doc2.txt_2
Insert of existing embedding ID: doc2.txt_2


Added doc2.txt_2 - 1742099152.529095

Add of existing embedding ID: doc2.txt_3
Insert of existing embedding ID: doc2.txt_3


Added doc2.txt_3 - 1742099152.529095

[{'file_name': 'doc5.txt', 'timestamp': 1743090118.542274, 'ids': [0, 1, 2]}, {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]}, {'file_name': 'doc1.txt', 'timestamp': 1742099250.607932, 'ids': [0, 1, 2, 3, 4]}, {'file_name': 'doc3.txt', 'timestamp': 1743090121.1379087, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc2.txt', 'timestamp': 1742099152.529095, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc5.txt', 'timestamp': 1743090118.542274, 'ids': [0, 1, 2]}, {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]}, {'file_name': 'doc1.txt', 'timestamp': 1742099250.607932, 'ids': [0, 1, 2, 3, 4]}, {'file_name': 'doc3.txt', 'timestamp': 1743090121.1379087, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc2.txt', 'timestamp': 1742099152.529095, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc5.txt', 'timestamp': 1743090118.542274, 'ids': [0, 1, 2]}, {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]}, {'file_name': 'doc1.txt', 'timestamp': 1742099250

In [38]:
[{'file_name': 'doc5.txt', 'timestamp': 1743089790.7844837, 'ids': [0, 1, 2]}, {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]}, {'file_name': 'doc1.txt', 'timestamp': 1742099250.607932, 'ids': [0, 1, 2, 3, 4]}, {'file_name': 'doc3.txt', 'timestamp': 1742099180.0197358, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc2.txt', 'timestamp': 1742099152.529095, 'ids': [0, 1, 2, 3]}]

[{'file_name': 'doc5.txt', 'timestamp': 1743089790.7844837, 'ids': [0, 1, 2]},
 {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]},
 {'file_name': 'doc1.txt',
  'timestamp': 1742099250.607932,
  'ids': [0, 1, 2, 3, 4]},
 {'file_name': 'doc3.txt',
  'timestamp': 1742099180.0197358,
  'ids': [0, 1, 2, 3]},
 {'file_name': 'doc2.txt',
  'timestamp': 1742099152.529095,
  'ids': [0, 1, 2, 3]}]

In [36]:
[{'file_name': 'doc5.txt', 'timestamp': 1742099200.6877513, 'ids': [0, 1, 2]}, {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]}, {'file_name': 'doc1.txt', 'timestamp': 1742099250.607932, 'ids': [0, 1, 2, 3, 4]}, {'file_name': 'doc3.txt', 'timestamp': 1742099180.0197358, 'ids': [0, 1, 2, 3]}, {'file_name': 'doc2.txt', 'timestamp': 1742099152.529095, 'ids': [0, 1, 2, 3]}]

[{'file_name': 'doc5.txt', 'timestamp': 1742099200.6877513, 'ids': [0, 1, 2]},
 {'file_name': 'doc4.txt', 'timestamp': 1742099190.638807, 'ids': [0, 1, 2]},
 {'file_name': 'doc1.txt',
  'timestamp': 1742099250.607932,
  'ids': [0, 1, 2, 3, 4]},
 {'file_name': 'doc3.txt',
  'timestamp': 1742099180.0197358,
  'ids': [0, 1, 2, 3]},
 {'file_name': 'doc2.txt',
  'timestamp': 1742099152.529095,
  'ids': [0, 1, 2, 3]}]